In [1]:
!pip install -q -U "transformers==4.51.3" "peft==0.15.2" "trl==0.17.0" "accelerate==1.6.0" "bitsandbytes==0.45.5" datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.6 MB/s eta 0:00:00:00:01


## Imports

In [2]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from peft import LoraConfig,PeftModel
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer,SFTConfig

## Quantization

In [3]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side="right"

#get the base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"":0}
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
data=[]
with open("/kaggle/input/datasets/koushikikundu/hr-policy-qs/hr_policy_synthetic_qa.jsonl", "r") as files:
    for file in files:
        d=json.loads(file)
        for i in d:
            for j in (i['conversations']):
                data.append(j['messages'])

In [5]:
def format(msg):
    return {
        "text": tokenizer.apply_chat_template(
            msg,
            tokenize=False,
            add_generation_prompt=False
        )
    }
for i in range(len(data)):
    data[i]=format(data[i])

In [6]:
data = Dataset.from_list(data)

In [7]:
print(data[0]['text'])

<s>[INST] Could you list some of the main programs offered by IIMA?[/INST] IIMA offers several major programs including the Two-year Post Graduate Programme in Management (MBA), Two-year Post Graduate Programme in Food and Agri-business Management (MBA - FABM), Ph.D. Programme in Management, One-year Post Graduate Programme in Management for Executives (MBA - PGPX), Faculty Development Programme for Teachers in Universities and Colleges (FDP), Armed Forces Programme for Officers for Indian Armed Forces (AFP), Two year ePost Graduate Programme (ePGP), and Sixteen months ePost Graduate Diploma in Advanced Business Analytics (ePGD - ABA).</s>


In [8]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [9]:
training_args = SFTConfig(
    output_dir="./mistral_hr",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    max_length=512,
    packing=True
)

In [10]:
trainer=SFTTrainer(
    model=model,
    train_dataset=data,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer
)

trainer.train()

Converting train dataset to ChatML:   0%|          | 0/1651 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1651 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1651 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1651 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,0.962000


TrainOutput(global_step=14, training_loss=0.9274086781910488, metrics={'train_runtime': 623.8085, 'train_samples_per_second': 0.357, 'train_steps_per_second': 0.022, 'total_flos': 4864946376302592.0, 'train_loss': 0.9274086781910488})

In [11]:
trainer.save_model("./mistral_hr")
tokenizer.save_pretrained("./mistral_hr")

('./mistral_hr/tokenizer_config.json',
 './mistral_hr/special_tokens_map.json',
 './mistral_hr/tokenizer.model',
 './mistral_hr/added_tokens.json',
 './mistral_hr/tokenizer.json')

In [12]:
fine_tuned_model = PeftModel.from_pretrained(
    model,
    "./mistral_hr"
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [14]:
prompt = f"""<s>[INST] can you give the list of the main programs offered by IIMA?? [/INST]"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda:0")

outputs = fine_tuned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.2,
    do_sample=False
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


can you give the list of the main programs offered by IIMA??  The Indian Institute of Management Ahmedabad (IIMA) offers a variety of programs. Here's a list of the main programs:

1. Post Graduate Program in Management (PGP)
2. Fellow Program in Management (FPM)
3. Executive Education Programs (EEPs)
4. Management Development Programs (MDPs)
5. Executive Development Programs (EDPs)
6. Customized Executive Development Programs (CEDPs)
